In [1]:
import cv2
import numpy as np

# Load YOLO model
net = cv2.dnn.readNet(
    "C:/Users/lenovo/Desktop/IPCV/yolov3-tiny.weights",
    "C:/Users/lenovo/Desktop/IPCV/yolov3-tiny.cfg"
)

# Load class names
with open("C:/Users/lenovo/Desktop/IPCV/coco.names", "r") as f:
    classes = f.read().splitlines()

# Load image
img = cv2.imread("C:/Users/lenovo/Desktop/IPCV/images/person.jpg")

if img is None:
    print("❌ Image not found. Check path.")
    exit()

height, width = img.shape[:2]

# Create blob
blob = cv2.dnn.blobFromImage(img, 1/255, (416, 416), swapRB=True, crop=False)
net.setInput(blob)

# Get output layers (no version issue)
output_layers = net.getUnconnectedOutLayersNames()

# Forward pass
outputs = net.forward(output_layers)

boxes = []
confidences = []
class_ids = []

# Process detections
for out in outputs:
    for det in out:
        scores = det[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]

        if confidence > 0.5:
            cx, cy, w, h = (det[0:4] * [width, height, width, height]).astype("int")
            x = int(cx - w / 2)
            y = int(cy - h / 2)

            boxes.append([x, y, w, h])
            confidences.append(float(confidence))
            class_ids.append(class_id)

# Apply Non-Max Suppression
indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

# Draw results
if len(indexes) > 0:
    for i in indexes.flatten():
        x, y, w, h = boxes[i]
        label = classes[class_ids[i]]
        conf = confidences[i]

        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(img, f"{label} {conf:.2f}",
                    (x, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 255, 0),
                    2)

# ✅ FINAL OUTPUT (AS YOUR MAM EXPECTS)
cv2.imshow("YOLO Detection", img)
cv2.waitKey(0)
cv2.destroyAllWindows()